# theoryminer — tutorial notebook

[Open this notebook in Google Colab](https://colab.research.google.com/github/rasoulnorouzi/theoryminer/blob/main/tutorials/TUTORIAL.ipynb)

This notebook is the tutorial and the reference of `theoryminer`. Each section shows one function,
its settings, its output shape, and the printed summary. Section 1.5 finds the DOI of each paper with
`extract_dois`, and section 1.6 finds the DOIs in their references with `extract_references`. Section 10 runs the whole pipeline on the sample papers, DOIs included, and
section 10.8 draws the causal map. Section 11 runs one snowball round: from one seed paper, through its
references and their downloads, to a new map. It runs in Google Colab, from `tutorials/`, or from the repository root. The examples use the open-access papers in
`sample_files/` (see `sample_files/SOURCES.md`). The slow step (`causenet` on all sample
sentences, a few minutes on CPU) loads a saved file when one exists.

The pipeline:

```
PDF ──extract_dois──► the DOI and title of each paper (a list, or a CSV file)
 ├──extract_references──► the DOIs that the papers cite ──download_papers──► the next round of PDFs
 │
 └──harvest──► sentences ──causenet──► relations (cause span → effect span)
                                              │
                    ┌─────────────────────────┼─────────────────────────┐
             standardize_constructs    pairwise_grouping            clusterer
             span → taxonomy concept   span → group (similarity)   span → cluster (density)
                    │                        └──── standardize_groups ────┘
                    │                               group → taxonomy concept
                    └─────────────────────────┬─────────────────────────┘
                                              ▼
                            label_groups / standardize_* : the name of each node
                                              │
                                              ▼
                  causal_map ──► draw_map (the interactive page), save_map (tables, GraphML)
```

The functions return their results in variables. Only five functions write files: `extract_dois` and
`extract_references` with `save=`, `download_papers` (the PDFs), `draw_map` (in a script, or with
`save=`) and `save_map`. Write other files yourself
when you need them (section 8).

## 0. Setup

Run the next cell first. It does three things:

1. It changes to the repository root when the notebook runs inside `tutorials/`.
2. It installs `theoryminer` from PyPI with pip when the package is missing. This is the case in a
   new Google Colab session.
3. It downloads the sample papers from GitHub when the folder `sample_files/` is missing.

In a clone of the repository with `pip install -e .`, the cell installs and downloads nothing.
In Colab, a GPU runtime makes `causenet` and the embeddings faster. A CPU runtime also works.

In [1]:
import importlib
import json
import os
import subprocess
import sys
import urllib.request

REPO = "rasoulnorouzi/theoryminer"
PIP_TARGET = "theoryminer"                # the latest release on PyPI; UMAP is included


def download_sample_files(folder="sample_files"):
    """Download every file of sample_files/ from GitHub into the folder."""
    listing_url = f"https://api.github.com/repos/{REPO}/contents/sample_files"
    with urllib.request.urlopen(listing_url) as response:
        listing = json.load(response)
    os.makedirs(folder, exist_ok=True)
    for item in listing:
        urllib.request.urlretrieve(item["download_url"], os.path.join(folder, item["name"]))
    print(f"downloaded {len(listing)} files into {folder}/")


# 1. Jupyter starts a notebook in its own folder. The paths below start at the repository root.
if os.path.basename(os.getcwd()) == "tutorials":
    os.chdir("..")

# 2. Install the package when pip has no record of it, as in a new Colab session.
# Ask pip's records, not the import system: an old cache folder alone can look like an
# empty "theoryminer" package, and an import would keep that empty package in memory.
try:
    importlib.metadata.version("theoryminer")
except importlib.metadata.PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIP_TARGET], check=True)
    importlib.invalidate_caches()

# 3. Get the sample papers when this folder is not a clone of the repository.
if not os.path.isdir("sample_files"):
    download_sample_files()

print("working folder:", os.getcwd())

working folder: /home/rass/Desktop/projects/theoryminer


In [2]:
import collections, json, os

from theoryminer import harvest, causenet, DEFAULT_STEPS, STEPS
from theoryminer.harmonizer import (
    standardize_constructs, standardize_groups, pairwise_grouping, clusterer, label_groups)

SAMPLE = "sample_files"                     # a folder of PDF files
ONE_PDF = os.path.join(SAMPLE, sorted(n for n in os.listdir(SAMPLE) if n.endswith(".pdf"))[0])
os.makedirs("outputs", exist_ok=True)
print("one PDF:", ONE_PDF)

one PDF: sample_files/alrabai_2021_autonomy_supportive_teaching.pdf


## 1. `harvest` — PDF to clean sentences

```python
data = harvest(source, pages=None, skip_pages=None, steps=None, min_words=4, max_words=80, cache=True)
```

`source` is one PDF file or a folder of PDF files. The function returns one dict with two lists:
`sentences` (kept rows) and `dropped` (the drop log). Each kept row has `doc_id`, `page`,
`sent_id`, `section`, `raw`, `clean`, `flag`. Each dropped row has `drop_reason`. The first call
extracts the page text with `pdf_inspector` and saves it next to each PDF as `<pdf>.pages.json`.
Later calls are fast.

In [3]:
data = harvest(SAMPLE)
print()
print("keys:", list(data))
print("first kept row:", data["sentences"][0])

  reading .pdf file 5 of 7

  reading .pdf file 7 of 7
harvest: sample_files
  files: 8 | read: 7 | skipped: 1
    skipped: SOURCES.md: the file name does not end with .pdf (not_pdf)
  text: 2,531 sentences kept | 1,865 rows dropped
       423  table_row
       328  no_lowercase
       247  reference_section
       229  too_short
       148  low_alpha
       107  running_head
       106  heading
        74  digit_heavy
        72  spaced_letters
        52  caption_or_keywords
        31  ref_fragment
        16  licence_text
        13  index_entry
         8  doi_url
         7  too_long
         2  empty_page
         1  page_number
         1  front_back_matter
  sentences per file:
       265  alrabai_2021_autonomy_supportive_teaching
       294  barrech_2018_job_insecurity_health
       296  friesinger_2025_discrimination_health
       384  gordesli_2024_social_media_mental_health
       318  pelikan_2021_needs_intrinsic_motivation
       611  schutz_2025_loneliness_social_support
       363  stephany_2017_

### 1.0 One file or a folder

For a folder, `harvest` reads each PDF on its own and then joins the results. The steps never
mix two papers. Each `sent_id` starts with the document number: `d002p0014s001` is document 2,
paragraph 14, sentence 1. So a `sent_id` is unique in the whole folder.

A bad file does not stop a folder run. It gets one row in the drop log with `page` 0:
`not_pdf` for a file whose name does not end with `.pdf`, `unreadable_pdf` for a PDF that
`pdf_inspector` cannot read or that has no text layer. `SOURCES.md` in the sample folder shows
the first case. For one PDF, a bad file raises a `ValueError`.

In [4]:
skipped = [r for r in data["dropped"] if r["page"] == 0]
print("skipped files:", skipped)
print()
single = harvest(ONE_PDF)
print()
print("sent_id of one PDF:", single["sentences"][0]["sent_id"])
print("doc_ids in the folder run:", sorted({r["doc_id"] for r in data["sentences"]}))

skipped files: [{'doc_id': 'SOURCES', 'page': 0, 'section': '', 'drop_reason': 'not_pdf', 'text': 'SOURCES.md: the file name does not end with .pdf'}]

harvest: sample_files/alrabai_2021_autonomy_supportive_teaching.pdf
  files: 1 | read: 1 | skipped: 0
  text: 265 sentences kept | 153 rows dropped
        47  reference_section
        34  running_head
        30  table_row
        23  heading
         8  too_short
         4  caption_or_keywords
         4  too_long
         2  no_lowercase
         1  digit_heavy

sent_id of one PDF: d000p0002s000
doc_ids in the folder run: ['alrabai_2021_autonomy_supportive_teaching', 'barrech_2018_job_insecurity_health', 'friesinger_2025_discrimination_health', 'gordesli_2024_social_media_mental_health', 'pelikan_2021_needs_intrinsic_motivation', 'schutz_2025_loneliness_social_support', 'stephany_2017_income_inequality_trust']


### 1.1 What `strip_citations` and `strip_stats` do: `raw` against `clean`

In [5]:
shown = 0
for r in data["sentences"]:
    if r["raw"] != r["clean"] and 60 < len(r["raw"]) < 260:
        print("RAW  :", r["raw"]); print("CLEAN:", r["clean"]); print()
        shown += 1
    if shown == 3: break

RAW  : In another definition, Benson (2001) perceived autonomy as the capacity to take charge of one's own leaning based on his desire, ability, and degree of freedom.
CLEAN: In another definition, Benson perceived autonomy as the capacity to take charge of one's own leaning based on his desire, ability, and degree of freedom.

RAW  : Autonomy-supportive teaching, according to Reeve (2016), is the delivery of instruction through an interpersonal tone of understanding that appreciates, supports, and vitalizes students' psychological needs.
CLEAN: Autonomy-supportive teaching, according to Reeve, is the delivery of instruction through an interpersonal tone of understanding that appreciates, supports, and vitalizes students' psychological needs.

RAW  : In this regard, Ikonen (2013) emphasized that the movement toward autonomy-supportive teaching in institutional settings has proven to fit surprisingly well.
CLEAN: In this regard, Ikonen emphasized that the movement toward autonomy-suppor

### 1.2 Drop reasons

Each dropped row carries the rule that removed it. The drop log sets the recall denominator.
`licence_text` (new in 0.2.0) is the licence and copyright text of a journal, for example
"Open Access This article is licensed under a Creative Commons Attribution 4.0 International License".
It is legal text, not research text, so the model does not read it.

In [6]:
reasons = collections.Counter(r["drop_reason"] for r in data["dropped"])
for reason, n in reasons.most_common():
    print(f"  {n:>6}  {reason}")
print()
flags = collections.Counter(r["flag"] for r in data["sentences"])
print("flags on kept rows:", dict(flags))

     423  table_row
     328  no_lowercase
     247  reference_section
     229  too_short
     148  low_alpha
     107  running_head
     106  heading
      74  digit_heavy
      72  spaced_letters
      52  caption_or_keywords
      31  ref_fragment
      16  licence_text
      13  index_entry
       8  doi_url
       7  too_long
       2  empty_page
       1  not_pdf
       1  page_number
       1  front_back_matter

flags on kept rows: {'fragment': 166, '': 2352, 'front_matter': 13}


### 1.2b Flags, and how to leave flagged sentences out

A flag marks a doubtful sentence. The sentence stays in `sentences`. The flags are:

| flag | meaning |
|---|---|
| `front_matter` | the PDF extractor mixed journal sidebar text into the sentence: an editor name, a date, an e-mail address |
| `fragment` | the sentence has no final punctuation |
| `long` | the sentence has more than 80 words |

To keep a flagged sentence away from the model, filter the list before `causenet`:

In [7]:
front = [s for s in data["sentences"] if s["flag"] == "front_matter"]
print(len(front), "sentences flagged front_matter; the first one:")
print("  ", front[0]["clean"][:160])

without_front_matter = [s for s in data["sentences"] if s["flag"] != "front_matter"]
print(len(data["sentences"]), "->", len(without_front_matter), "sentences")

13 sentences flagged front_matter; the first one:
   Strategies targeting learners' self-determined learning in the classroom in terms of satisfying learner basic psychological needs (BPNs) of autonomy, competence
2531 -> 2518 sentences


### 1.3 The steps list

Cleaning is an ordered list of steps. Remove a name, change the order, or insert your own
function. The contract for each step: take a list of row dicts, return a list of row dicts. To
drop a row, set `kept=False` and a `drop_reason`. Do not delete the row.

In [8]:
print(DEFAULT_STEPS)

def no_questions(rows):
    """Drop sentences that end with a question mark."""
    for r in rows:
        if r["kept"] and r["text"].rstrip().endswith("?"):
            r["kept"] = False
            r["drop_reason"] = "question"
    return rows

custom = harvest(ONE_PDF,
                 steps=["fix_unicode", "strip_markdown", "drop_junk_lines", "join_paragraphs",
                        "split_sentences", no_questions, "filter_length"])
print()
print("dropped as question:", [r["text"] for r in custom["dropped"] if r["drop_reason"] == "question"][:3])

['fix_unicode', 'strip_markdown', 'drop_running_heads', 'drop_junk_lines', 'join_paragraphs', 'split_sentences', 'strip_citations', 'strip_stats', 'drop_junk_sentences', 'filter_length']
harvest: sample_files/alrabai_2021_autonomy_supportive_teaching.pdf
  files: 1 | read: 1 | skipped: 0
  text: 295 sentences kept | 122 rows dropped
        50  reference_section
        30  table_row
        23  heading
        10  too_short
         6  too_long
         3  question

dropped as question: ['RQ1: the autonomy of Saudi EFL learners?', "What are the most important strategies to promote RQ2: enhance EFL learner's autonomy?", 'Will teacher autonomy-supportive teaching RQ2: would causally predict learner EFL autonomy?']


### 1.4 Page control

`pages` keeps a window. `skip_pages` removes pages inside it.

In [9]:
small = harvest(ONE_PDF, pages="2-6", skip_pages=[4])
print()
print("pages seen:", sorted({r["page"] for r in small["sentences"]}))

harvest: sample_files/alrabai_2021_autonomy_supportive_teaching.pdf
  files: 1 | read: 1 | skipped: 0
  text: 82 sentences kept | 26 rows dropped
        11  table_row
         5  heading
         4  running_head
         3  too_long
         1  digit_heavy
         1  too_short
         1  caption_or_keywords

pages seen: [2, 3, 5, 6]


### 1.5 The DOI of each paper: `extract_dois`

```python
rows = extract_dois(source, save=None, first_pages=2, cache=True)
```

`source` is one PDF or a folder, as in `harvest`. A paper holds many DOIs, most of them in its
reference list. The DOI of the paper itself is at the top of page 1: in the header, the citation box
or the footer. So the rule is: the first DOI on page 1, else the first DOI on page 2. It was checked by
hand on the 7 sample papers: 7 of 7 correct.

The result is one row per file: `doc_id`, `file`, `doi`, `title`, `page`, `status`, `note`.
The `title` (new in 0.2.0) is the text in the largest font on page 1 that has at least 4 words. A
shorter large text is a journal banner, such as "PLOS ONE". When no text passes, the title field of the
PDF metadata is used. On the 7 sample papers the rule found 7 of 7 titles. `draw_map` shows the title
of a paper that has no DOI. The `status` is
`found`, `not_found`, `not_pdf` or `unreadable_pdf`. A bad file does not stop the run.
`save="outputs"` writes `outputs/dois.csv`. `save="outputs/my_dois.csv"` writes that file.

In [10]:
from theoryminer import extract_dois

rows = extract_dois(SAMPLE, save="outputs")
print()
for row in rows:
    print(f"  {row['status']:<10} {row['doi']:<30} {row['file']}")

  reading .pdf file 7 of 7


dois: sample_files
  files: 8 | found: 7 | not found: 0 | skipped: 1
    not_pdf: SOURCES.md: the file name does not end with .pdf
  saved: outputs/dois.csv

  not_pdf                                   SOURCES.md
  found      10.3389/fpsyg.2021.728657      alrabai_2021_autonomy_supportive_teaching.pdf
  found      10.1186/s12889-018-5621-4      barrech_2018_job_insecurity_health.pdf
  found      10.1186/s12889-025-23888-6     friesinger_2025_discrimination_health.pdf
  found      10.1371/journal.pone.0316365   gordesli_2024_social_media_mental_health.pdf
  found      10.1371/journal.pone.0257346   pelikan_2021_needs_intrinsic_motivation.pdf
  found      10.1186/s12889-025-23247-5     schutz_2025_loneliness_social_support.pdf
  found      10.1007/s11205-016-1460-9      stephany_2017_income_inequality_trust.pdf


### 1.6 The DOIs in the references: `extract_references`

```python
refs = extract_references(source, save=None, cache=True)
```

Give it the seed papers: one PDF or a folder. It reads every page of each paper and finds each DOI.
The seed's own DOI and the DOIs of its parts (figures and tables, such as `….g001`) are left out. The
result has one row per unique DOI: `doi`, `n_seeds` (how many seeds cite it) and `cited_by` (their
`doc_id`s). The DOIs that most seeds cite come first: these are the best candidates for the next round
of a snowball search. `save="outputs"` writes `outputs/references.csv`, with the `doi` column first.

A PDF often cuts a DOI at a line end. The function joins the parts again:

| the PDF text | the DOI |
|---|---|
| `10.1007/s1096 4-015-0354-5` | `10.1007/s10964-015-0354-5` |
| `10.3389/fpubh` + line break + `.2024.1392999` | `10.3389/fpubh.2024.1392999` |
| `10.1 186/s12888-020-02818-3` | `10.1186/s12888-020-02818-3` |
| `10.1177%2F1362168817725759` | `10.1177/1362168817725759` |

A year ("2021"), a word ("Accessed") and the number of the next reference ("36. Smith J") are never
joined. On the 7 sample papers the function finds 215 reference DOIs, and 214 of them exist at doi.org.
The one miss is printed without its last digit in the PDF itself. A reference that prints no DOI cannot
be found: the 2018 paper of Barrech and colleagues has none.

In [11]:
from theoryminer import extract_references

refs = extract_references(SAMPLE, save="outputs")
print()
print("the most cited:", refs[0])
print()
for row in refs[1:6]:
    print(f"  {row['doi']:<40} cited by {row['cited_by']}")

  reading .pdf file 7 of 7
references: sample_files
  seeds read: 7 | unique reference DOIs: 215 | cited by 2 or more seeds: 1
       41  alrabai_2021_autonomy_supportive_teaching
        0  barrech_2018_job_insecurity_health
        3  friesinger_2025_discrimination_health
       50  gordesli_2024_social_media_mental_health
       42  pelikan_2021_needs_intrinsic_motivation
       74  schutz_2025_loneliness_social_support
        6  stephany_2017_income_inequality_trust
  no DOI in the references of: barrech_2018_job_insecurity_health (the journal printed none)
  skipped: SOURCES.md (the file name does not end with .pdf)
  saved: outputs/references.csv

the most cited: {'doi': '10.1207/s15327965pli1104_01', 'n_seeds': 2, 'cited_by': ['alrabai_2021_autonomy_supportive_teaching', 'pelikan_2021_needs_intrinsic_motivation']}

  10.1001/archinte.166.10.1092             cited by ['gordesli_2024_social_media_mental_health']
  10.1001/jamapsychiatry.2019.2325         cited by ['gordesli_2024_

To download the papers, give the result to `download_papers` (section 11). It checks the licence of each
paper with OpenAlex first, then downloads the open-access PDFs through OpenAlex and `tmsr-doi-downloader`.
`pip install theoryminer` installs that tool on Python 3.11 or newer. Its Unpaywall, CORE and Google
Scholar sources need keys: `email=` (your own email address; Unpaywall refuses a made-up one),
`core_api_key=` (free, from core.ac.uk) and `serpapi_key=` (a paid service). The environment variables
`UNPAYWALL_EMAIL`, `CORE_API_KEY` and `SERPAPI_KEY` also work. Not every paper can be downloaded: many
publishers block robots.

```python
report = download_papers(refs, "round_2", licences=["cc-by"])   # one status row per DOI
round_2 = harvest("round_2")                                       # the next round starts here
```

## 2. `causenet` — sentences to cause-effect pairs

```python
relations = causenet(sentences, threshold=0.8, mode="neural", decision="span_only", batch_size=64,
                     device="auto", avoid_ambiguous=True)
```

`decision="span_only"` (the default since 0.2.0) takes a sentence as causal when the model finds a
cause span and an effect span. `"cls+span"` (the default before 0.2.0) also asks the sentence
classifier. `avoid_ambiguous=True` skips a relation whose cause or effect only points to another
sentence: "this", "it", "such things". The summary prints how many relations it skipped, with
examples. `avoid_ambiguous=False` keeps them.

The model is `rasoultilburg/SocioCausaNet`. It loads once per session. `device="auto"` runs it on
an NVIDIA GPU when torch can see one, else on an Apple Silicon GPU, else on the CPU. Give `"cpu"`,
`"cuda"` or `"mps"` to choose. On a
GPU, a larger `batch_size` is usually faster. It returns one dict per
relation: `rel_id`, `sent_id`, `doc_id`, `page`, `cause`, `effect`, `sentence`. `rel_id` is
`sent_id` + `r<n>`, the n-th relation of the sentence. The spans are substrings of the sentence.
Nothing paraphrases them. A cut span such as `'teachers ( i.'` stays as it is: the span is the
provenance key.

In [12]:
causenet(["Smoking causes lung cancer and heart disease",
          "Exercise improves physical health and mental well-being",
          "This leads to higher stress among employees",      # the cause only points elsewhere: skipped
          "This is a non-causal sentence about weather"])

/home/rass/Desktop/projects/theoryminer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


causenet: loading rasoultilburg/SocioCausaNet ...


Loading weights:   0%|          | 0/217 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 217/217 [00:00<00:00, 4294.44it/s]

causenet: device = cpu


  4/4 sentences (1s)
causenet: 3 causal sentences of 4 | 4 relations
  spans repaired after decoding: 0 | both spans found verbatim in their sentence: 100%
  skipped (avoid_ambiguous=True): 1 relations with a pointer span, for example: this -> higher stress among employees


[{'rel_id': 's0000r00',
  'sent_id': 's0000',
  'doc_id': 'input',
  'page': 0,
  'cause': 'smoking',
  'effect': 'lung cancer',
  'sentence': 'Smoking causes lung cancer and heart disease'},
 {'rel_id': 's0000r01',
  'sent_id': 's0000',
  'doc_id': 'input',
  'page': 0,
  'cause': 'smoking',
  'effect': 'heart disease',
  'sentence': 'Smoking causes lung cancer and heart disease'},
 {'rel_id': 's0001r00',
  'sent_id': 's0001',
  'doc_id': 'input',
  'page': 0,
  'cause': 'exercise',
  'effect': 'physical health',
  'sentence': 'Exercise improves physical health and mental well-being'},
 {'rel_id': 's0001r01',
  'sent_id': 's0001',
  'doc_id': 'input',
  'page': 0,
  'cause': 'exercise',
  'effect': 'mental well-being',
  'sentence': 'Exercise improves physical health and mental well-being'}]

### 2.1 A real run: all sample papers

A few minutes on CPU. The cell loads the saved file when it exists.

In [13]:
REL_FILE = "outputs/relations_sample.json"
if os.path.exists(REL_FILE):
    relations = json.load(open(REL_FILE))
    print(f"loaded {len(relations):,} relations from {REL_FILE}")
else:
    relations = causenet(data["sentences"])
    json.dump(relations, open(REL_FILE, "w"))

print()
print("relations[0]:", relations[0])
print()
for r in relations[:6]:
    print(f"  [{r['doc_id'][:20]} p{r['page']} {r['rel_id']}] {r['cause']!r} -> {r['effect']!r}")

  64/2,531 sentences (5s)

  128/2,531 sentences (8s)

  192/2,531 sentences (10s)

  256/2,531 sentences (13s)

  320/2,531 sentences (16s)

...

  2,304/2,531 sentences (107s)

  2,368/2,531 sentences (112s)

  2,432/2,531 sentences (114s)

  2,496/2,531 sentences (115s)

  2,531/2,531 sentences (116s)
causenet: 490 causal sentences of 2,531 | 945 relations
  spans repaired after decoding: 124 | both spans found verbatim in their sentence: 97%
  skipped (avoid_ambiguous=True): 44 relations with a pointer span, for example: these differences -> models being more comprehensive; powerful enough to disclose the net effect size of the treatment manipulated in this study -> this; it -> impacts individuals'health and well-being

relations[0]: {'rel_id': 'd000p0003s003r00', 'sent_id': 'd000p0003s003', 'doc_id': 'alrabai_2021_autonomy_supportive_teaching', 'page': 1, 'cause': 'experimental intervention', 'effect': 'efl ehsan namaziandost', 'sentence': 'The Reviewed by: findings derived out from analyses of variance, covariance and a hierarchical regression Zhengdong Gan, University of Macau, China revealed that the experimental intervention led to statistically significant increased EFL Ehsan Namaziandost, autonomy for learners in the experimental group.'}

  [a

### 2.2 The threshold

A lower threshold gives more pairs. The old pipeline swept 0.5–0.8 and adopted 0.8.

In [14]:
sample = [r["sentence"] for r in relations[:40]]
strict = causenet(sample, threshold=0.8)
loose  = causenet(sample, threshold=0.5)
print()
print(f"threshold 0.8: {len(strict)} relations | threshold 0.5: {len(loose)} relations")

  40/40 sentences (3s)
causenet: 40 causal sentences of 40 | 242 relations
  spans repaired after decoding: 66 | both spans found verbatim in their sentence: 99%


  40/40 sentences (4s)
causenet: 40 causal sentences of 40 | 268 relations
  spans repaired after decoding: 89 | both spans found verbatim in their sentence: 98%

threshold 0.8: 242 relations | threshold 0.5: 268 relations


### 2.3 One construct, two roles

A span that is a cause in one sentence and an effect in another is one node. Chains pass through these spans only.

In [15]:
causes  = collections.Counter(r["cause"] for r in relations)
effects = collections.Counter(r["effect"] for r in relations)
both = sorted((s for s in causes if s in effects), key=lambda s: -(causes[s] + effects[s]))
print(f"{len(causes):,} distinct causes | {len(effects):,} distinct effects | {len(both)} spans in both roles")
for s in both[:6]:
    print(f"  {s!r}: cause x{causes[s]}, effect x{effects[s]}")
s = both[0]
print()
print("as CAUSE :", next(f"[{r['rel_id']}] {r['sentence']}" for r in relations if r["cause"] == s))
print("as EFFECT:", next(f"[{r['rel_id']}] {r['sentence']}" for r in relations if r["effect"] == s))

563 distinct causes | 675 distinct effects | 41 spans in both roles
  'social support': cause x20, effect x2
  'loneliness': cause x15, effect x6
  'job insecurity': cause x8, effect x4
  'autonomy': cause x4, effect x4
  'health': cause x1, effect x6
  'pronounced quest for autonomy': cause x6, effect x1

as CAUSE : [d005p0013s011r00] Adapting and expanding this main effect model to address loneliness, social support may fulfill the basic human needs for connection, belonging, and mutual understanding [54].
as EFFECT: [d005p0077s002r01] The importance of strengthening social relationships and social support can be deduced from the findings that both loneliness and mental health problems are significantly less pronounced in children and adolescents who experience a high level of social support.


## 3. The harmonizer — one idea, three routes

Harmonization decides which spans are the **same node**. All three routes take the `relations`
list (or a plain list of strings), pool causes and effects into one set of unique spans, and
return a dict keyed by the original span. Vectors are L2-normalized in one place, so each
similarity is a cosine.

| route | function | node identity by |
|---|---|---|
| taxonomy | `standardize_constructs` | closeness to a thesaurus concept |
| pairwise | `pairwise_grouping` | closeness to other spans (threshold, connected components) |
| density | `clusterer` | HDBSCAN cluster |
| taxonomy for groups | `standardize_groups` | closeness of a whole group to a concept |

Models: `allmpnet` (default), `bge_base`, `minilm`, or any sentence-transformers id.

`tidy_span` repairs what the tokenizer breaks when the model decodes a span. `causenet`
applies it once. It removes no word. No function removes function words: the models handle
them, and words such as *lack*, *not*, *low* carry the polarity of a construct.

In [16]:
from theoryminer.harmonizer.spans import collect_spans, tidy_span
spans = collect_spans(relations)
print(len(spans), "unique spans; first five:", spans[:5])
for q in ["society ' s desired values", "[CLS] autonomy support is critical", "inte ##grative processes",
          "), and intrapersonal events ( self-set goals )", "intrinsic motivation,", "the lack of autonomy"]:
    print(f"  tidy_span({q!r}) -> {tidy_span(q)!r}")
verbatim = sum(r["cause"].lower() in r["sentence"].lower() and r["effect"].lower() in r["sentence"].lower()
               for r in relations)
print(f"both spans found verbatim in their sentence: {verbatim / len(relations):.0%}")

1197 unique spans; first five: ['experimental intervention', 'efl ehsan namaziandost', 'desire', 'autonomy', 'take charge']
  tidy_span("society ' s desired values") -> "society's desired values"
  tidy_span('[CLS] autonomy support is critical') -> 'autonomy support is critical'
  tidy_span('inte ##grative processes') -> 'integrative processes'
  tidy_span('), and intrapersonal events ( self-set goals )') -> 'and intrapersonal events (self-set goals)'
  tidy_span('intrinsic motivation,') -> 'intrinsic motivation'
  tidy_span('the lack of autonomy') -> 'the lack of autonomy'
both spans found verbatim in their sentence: 97%


## 4. `standardize_constructs` — spans to taxonomy concepts

```python
matches = standardize_constructs(relations, taxonomy="elsst", strategy="enriched", top=5,
                                 embeddings="allmpnet", leaves_only=False, cache=True)
```

Returns `{span: [{"id", "leaf", "path", "score"}, ...]}`, best first, one item per concept. No
threshold is applied. τ is a downstream decision.

In [17]:
matches = standardize_constructs(relations, strategy="enriched", top=5)
print()
for span in list(matches)[:6]:
    print(f"{span:<62}", " | ".join(f"{m['leaf']} ({m['score']:.2f})" for m in matches[span][:3]))
print()
print("full record for one span:", matches[list(matches)[0]][0])

taxonomy: 3,436 concepts | 2,417 leaves | 3,739 entries (all concepts)
taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)


embeddings: loading sentence-transformers/all-mpnet-base-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10019.40it/s]

standardize: 1,197 spans -> top 5 of 3,739 entries | best score: median 0.51, min 0.16, max 0.83

experimental intervention                                      PSYCHOLOGICAL TESTS (0.44) | DEMONSTRATIONS (LESSONS) (0.40) | RESEARCH (0.40)
efl ehsan namaziandost                                         IDENTITY (0.28) | INTERNATIONAL SANCTIONS (0.25) | ELSST Thesaurus (0.25)
desire                                                         MOTIVATION (0.57) | ASPIRATION (0.43) | CONSUMPTION (0.41)
autonomy                                                       AUTONOMY AT WORK (0.52) | RIGHTS AND PRIVILEGES (0.47) | IDENTITY (0.43)
take charge                                                    POLITICAL ACTION (0.41) | MOTIVATION (0.39) | ROLES (0.36)
s own leaning                                                  ATTITUDES (0.38) | PREJUDICE (0.36) | ATTITUDE CHANGE (0.33)

full record for one span: {'id': 'https://elsst.cessda.eu/id/5/9ecc4359-36f2-495a-a685-5bf729329efd', 'leaf': 'PSYCHOL

### 4.1 The score distribution

Read it before you choose τ. Many spans sit far from any ELSST concept. That is a fact about the thesaurus.

In [18]:
best = sorted(m[0]["score"] for m in matches.values())
n = len(best)
print(f"n={n:,}  p10={best[n//10]:.2f}  median={best[n//2]:.2f}  p90={best[9*n//10]:.2f}")
for tau in (0.50, 0.55, 0.60, 0.70):
    print(f"  share >= {tau}: {sum(b >= tau for b in best) / n:.1%}")
by_concept = collections.Counter(m[0]["leaf"] for m in matches.values())
print()
print(f"{n:,} spans -> {len(by_concept)} distinct top-1 concepts; largest:", by_concept.most_common(4))

n=1,197  p10=0.36  median=0.51  p90=0.66
  share >= 0.5: 52.9%
  share >= 0.55: 35.2%
  share >= 0.6: 19.4%
  share >= 0.7: 5.5%

1,197 spans -> 436 distinct top-1 concepts; largest: [('LONELINESS', 60), ('LEARNING', 38), ('TRUST', 34), ('PHYSIOLOGICAL EFFECTS', 27)]


### 4.2 `leaves_only`: where a span may land

A **leaf** has no narrower concept. A **branch** has children. `leaves_only=False` (the default
since 0.2.0) lets a span land on a leaf or on a branch, so a broad construct can get a broad concept
("TRUST", "MOTIVATION"). `leaves_only=True` (the default before 0.2.0) lets it land on leaves only. The switch changes the set
of possible answers, not a score. `d` is the depth of the concept in the tree. Full
explanation: `tutorials/STRATEGIES.md`.

In [19]:
probe = ["autonomy", "intrinsic motivation", "well-being", "autonomy support",
         "loneliness", "unemployment", "social trust", "depressive symptoms"]
m_leaf = standardize_constructs(probe, leaves_only=True,  top=1)
m_all  = standardize_constructs(probe, leaves_only=False, top=1)
print()
for s in probe:
    a, b = m_leaf[s][0], m_all[s][0]
    print(f"  {s:<22} leaves: {a['leaf']:<22} d={a['path'].count('>') + 1} ({a['score']:.2f})   "
          f"all: {b['leaf']:<22} d={b['path'].count('>') + 1} ({b['score']:.2f})")

taxonomy: 3,436 concepts | 2,417 leaves | 2,661 entries (leaves only)
taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)
standardize: 8 spans -> top 1 of 2,661 entries | best score: median 0.60, min 0.46, max 0.69
taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)
standardize: 8 spans -> top 1 of 3,739 entries | best score: median 0.62, min 0.52, max 0.76

  autonomy               leaves: AUTONOMY AT WORK       d=3 (0.52)   all: AUTONOMY AT WORK       d=3 (0.52)
  intrinsic motivation   leaves: INCENTIVES             d=2 (0.65)   all: MOTIVATION             d=1 (0.69)
  well-being             leaves: HAPPINESS              d=3 (0.55)   all: WELL-BEING (HEALTH)    d=1 (0.61)
  autonomy support       leaves: AUTONOMY AT WORK       d=3 (0.57)   all: AUTONOMY AT WORK       d=3 (0.57)
  loneliness             leaves: LONELINESS             d=3 (0.63)   all: LONELINESS             d=3 (0.63)
  unemployment           leaves: UNEMPLOYED             d=3 (0.64)  

### 4.3 The six strategies

The strategy decides what text stands for a concept. `leaf` = the label. `path`, `anchor`,
`context`, `bracket` = the label with the tree in four phrasings. `enriched` = label, synonyms,
note, path. The winner rarely changes across strategies; the score does. A τ set for one
strategy is wrong for another.

In [20]:
from theoryminer.harmonizer.taxonomy import load_taxonomy, strategy_text, STRATEGIES
e = next(x for x in load_taxonomy("elsst", leaves_only=True) if x["leaf"] == "SELF-ESTEEM")
for s in STRATEGIES:
    print(f"  {s:<9} {strategy_text(e, s)}")
print()
for strategy in STRATEGIES:
    m = standardize_constructs(probe[:6], strategy=strategy, top=1)
    print(f"{strategy:<9}", " | ".join(f"{sp}: {m[sp][0]['leaf']} ({m[sp][0]['score']:.2f})" for sp in probe[:6]))

  leaf      self-esteem
  path      identity > personal identity > self-esteem
  anchor    self-esteem: identity > personal identity > self-esteem
  context   self-esteem is related to personal identity, identity
  bracket   self-esteem (personal identity, identity)
  enriched  self-esteem. self-acceptance; self-confidence; self-respect. identity > personal identity > self-esteem

taxonomy: no cache yet for (elsst, leaf, allmpnet, all). Embedding 3,739 entries now; about 3 min on CPU. This runs once; later calls load elsst_leaf_allmpnet_all.npy.


standardize: 6 spans -> top 1 of 3,739 entries | best score: median 0.79, min 0.72, max 1.00
leaf      autonomy: AUTONOMY AT WORK (0.73) | intrinsic motivation: MOTIVATION (0.77) | well-being: WELL-BEING (HEALTH) (0.82) | autonomy support: AUTONOMY AT WORK (0.72) | loneliness: LONELINESS (1.00) | unemployment: UNEMPLOYMENT (1.00)
taxonomy: no cache yet for (elsst, path, allmpnet, all). Embedding 3,739 entries now; about 3 min on CPU. This runs once; later calls load elsst_path_allmpnet_all.npy.


standardize: 6 spans -> top 1 of 3,739 entries | best score: median 0.64, min 0.57, max 0.82
path      autonomy: AUTONOMY AT WORK (0.57) | intrinsic motivation: MOTIVATION (0.77) | well-being: WELL-BEING (HEALTH) (0.82) | autonomy support: AUTONOMY AT WORK (0.59) | loneliness: LONELINESS (0.61) | unemployment: UNEMPLOYMENT (0.68)
taxonomy: no cache yet for (elsst, anchor, allmpnet, all). Embedding 3,739 entries now; about 3 min on CPU. This runs once; later calls load elsst_anchor_allmpnet_all.npy.


standardize: 6 spans -> top 1 of 3,739 entries | best score: median 0.68, min 0.62, max 0.73
anchor    autonomy: AUTONOMY AT WORK (0.64) | intrinsic motivation: MOTIVATION (0.73) | well-being: WELL-BEING (SOCIETY) (0.70) | autonomy support: AUTONOMY AT WORK (0.67) | loneliness: LONELINESS (0.62) | unemployment: UNEMPLOYMENT (0.69)
taxonomy: no cache yet for (elsst, context, allmpnet, all). Embedding 3,739 entries now; about 3 min on CPU. This runs once; later calls load elsst_context_allmpnet_all.npy.


standardize: 6 spans -> top 1 of 3,739 entries | best score: median 0.66, min 0.62, max 0.82
context   autonomy: AUTONOMY AT WORK (0.66) | intrinsic motivation: MOTIVATION (0.77) | well-being: WELL-BEING (HEALTH) (0.82) | autonomy support: AUTONOMY AT WORK (0.65) | loneliness: LONELINESS (0.62) | unemployment: UNEMPLOYMENT (0.65)
taxonomy: no cache yet for (elsst, bracket, allmpnet, all). Embedding 3,739 entries now; about 3 min on CPU. This runs once; later calls load elsst_bracket_allmpnet_all.npy.


standardize: 6 spans -> top 1 of 3,739 entries | best score: median 0.71, min 0.63, max 0.82
bracket   autonomy: AUTONOMY AT WORK (0.63) | intrinsic motivation: MOTIVATION (0.77) | well-being: WELL-BEING (HEALTH) (0.82) | autonomy support: AUTONOMY AT WORK (0.65) | loneliness: LONELINESS (0.64) | unemployment: UNEMPLOYMENT (0.77)
taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)
standardize: 6 spans -> top 1 of 3,739 entries | best score: median 0.62, min 0.52, max 0.69
enriched  autonomy: AUTONOMY AT WORK (0.52) | intrinsic motivation: MOTIVATION (0.69) | well-being: WELL-BEING (HEALTH) (0.61) | autonomy support: AUTONOMY AT WORK (0.57) | loneliness: LONELINESS (0.63) | unemployment: UNEMPLOYMENT (0.69)


### 4.4 The cache

The first call for a (taxonomy, strategy, model, leaves/all) combination embeds all entries and
saves them in `~/.cache/theoryminer/taxonomy` (or in the folder in the environment variable
`THEORYMINER_CACHE_DIR`). If the folder cannot be written, the run goes on without the cache. Files are plain: `.npy` vectors and a
`.json` entry list in the same row order. R reads them with `RcppCNPy` and `jsonlite`.

In [21]:
from theoryminer.harmonizer.taxonomy import CACHE_DIR
import numpy as np
files = sorted(os.listdir(CACHE_DIR))
print(len(files), "files, e.g.", files[:4])
v = np.load(os.path.join(CACHE_DIR, "elsst_enriched_allmpnet_leaves.npy"))
entries = json.load(open(os.path.join(CACHE_DIR, "elsst_leaves.json"), encoding="utf-8"))
print("vectors:", v.shape, v.dtype, "| entries:", len(entries), "| row 0 =", entries[0]["leaf"])

14 files, e.g. ['elsst_all.json', 'elsst_anchor_allmpnet_all.npy', 'elsst_anchor_allmpnet_leaves.npy', 'elsst_bracket_allmpnet_all.npy']
vectors: (2661, 768) float32 | entries: 2661 | row 0 = ELSST Thesaurus


## 5. `pairwise_grouping` — spans to groups by similarity

```python
groups = pairwise_grouping(relations, embeddings="allmpnet", threshold=0.69, linkage="average")
```

`linkage` decides when a span joins a group:

| `linkage` | a span joins a group when its cosine … |
|---|---|
| `"average"` (the default since 0.2.0) | to the members, on average, is ≥ threshold |
| `"complete"` | to every member is ≥ threshold |
| `"single"` (the default before 0.2.0, Paper 3's method) | to one member is ≥ threshold |

`central` is the medoid. 0.69 is Paper 3's calibrated threshold for all-mpnet.

In [22]:
groups = pairwise_grouping(relations, threshold=0.69)
print()
by_group = collections.defaultdict(list)
for span, g in groups.items():
    by_group[g["group"]].append(span)
sizes = sorted(by_group.values(), key=len, reverse=True)
print("largest group:", len(sizes[0]), "| singletons:", sum(len(m) == 1 for m in sizes))
print()
print("a good group:", next(m for m in sizes if 4 <= len(m) <= 6))
print()
print("one span:", groups[list(groups)[0]])

pairwise: 1,197 spans -> 811 groups at cos >= 0.69, average linkage | 180 groups with 2+ members, largest 27

largest group: 27 | singletons: 631

a good group: ['perception of inequality', 'observed in an environment of overall high levels of inequality', 'inequality perception does not necessarily mirror actual inequalities', 'high perceived inequality', 'increased perceived inequality', 'closer to capturing the impact of perceived inequality']

one span: {'group': 0, 'central': 'fixing the paths'}


### 5.1 The chaining problem, and why the default is average linkage

Single linkage chains: A links B, B links C, so A and C are in one group, even when A and C are
unrelated. One component can then hold a large share of the spans. The cell makes the single-linkage
groups, finds one chain in the largest group, and compares the largest group of the three rules.
Average linkage stops the chain, because one close member is not enough.

In [23]:
from theoryminer.harmonizer.embeddings import embed

# 1. The single-linkage groups, and the largest one.
single = pairwise_grouping(relations, linkage="single")
by_single = collections.defaultdict(list)
for span, g in single.items():
    by_single[g["group"]].append(span)
big = max(by_single.values(), key=len)
print("largest single-linkage group, sample:", big[:8])

# 2. A chain in that group: A-B and B-C pass 0.69, A-C is the lowest.
vectors = embed(big)
similarity = vectors @ vectors.T
best = None
for i in range(len(big)):
    for j in range(len(big)):
        if i == j or similarity[i, j] < 0.69:
            continue
        for k in range(len(big)):
            if k in (i, j) or similarity[j, k] < 0.69:
                continue
            if best is None or similarity[i, k] < best[0]:
                best = (similarity[i, k], big[i], big[j], big[k], similarity[i, j], similarity[j, k])
if best is None:
    print("no chain of three spans in the largest group")
else:
    print(f"chain: {best[1]!r} -{best[4]:.2f}- {best[2]!r} -{best[5]:.2f}- {best[3]!r}; direct cosine {best[0]:.2f}")

# 3. The three rules at the default threshold.
print()
for rule in ("single", "average", "complete"):
    pairwise_grouping(relations, linkage=rule)

pairwise: 1,197 spans -> 659 groups at cos >= 0.69, single linkage | 90 groups with 2+ members, largest 148
largest single-linkage group, sample: ['relatedness', 'trust in the', 'experience discrimination', 'discrimination', 'reporting discrimination for one', 'individuals experiencing discrimination for three or more reasons', 'discrimination in more subtle ways', 'racism']


chain: 'social trust' -0.73- 'inequality and trust' -0.70- 'perception of inequality'; direct cosine 0.33



pairwise: 1,197 spans -> 659 groups at cos >= 0.69, single linkage | 90 groups with 2+ members, largest 148


pairwise: 1,197 spans -> 811 groups at cos >= 0.69, average linkage | 180 groups with 2+ members, largest 27


pairwise: 1,197 spans -> 856 groups at cos >= 0.69, complete linkage | 206 groups with 2+ members, largest 10


## 6. `clusterer` — spans to density clusters

```python
clusters = clusterer(relations, embeddings="allmpnet", min_cluster_size=2, min_samples=2, umap=None)
```

HDBSCAN on the normalized vectors. Defaults are Paper 3's calibrated configuration. Group −1 is
noise: a span with no near neighbour. It becomes a node of one. It is never dropped.

In [24]:
clusters = clusterer(relations)
print()
by_cluster = collections.defaultdict(list)
for span, g in clusters.items():
    by_cluster[g["group"]].append(span)
for cid, members in sorted(by_cluster.items(), key=lambda kv: -len(kv[1]))[:6]:
    if cid == -1: continue
    print(f"C{cid} [{clusters[members[0]]['central']}]  {members[:5]} ...")
print()
print("noise example:", clusters[by_cluster[-1][0]])
print("noise share:", f"{len(by_cluster[-1]) / len(clusters):.0%}")

/home/rass/Desktop/projects/theoryminer/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


cluster: 1,197 spans -> 242 clusters | 468 noise (39%) | raw embeddings

C223 [higher intrinsic motivation]  ['greater classroom motivation', "learner's intrinsic motivation was", 'intrinsic motivation', 'high intrinsic motivation', 'low intrinsic motivation'] ...
C124 [procrastination]  ['maladaptive behavior like procrastination', 'procrastinated less', 'procrastinate less', 'procrastination', 'procrastinated'] ...
C240 [promoting learner autonomy]  ['institutional setting is an appropriate environment for promoting learner autonomy', 'promote learner autonomy', 'development of learner autonomy', 'learner autonomy', 'multidimensionality of the concept of learner autonomy'] ...
C76 [distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful]  ["teachers'difficulties in adapting to remote teaching", 'distance learning', 'switched to distance learning', 'distance teaching requires ample preparation time and situation-specific didactic a

With a UMAP reduction first, for comparison. Less noise, but stochastic and not the calibrated configuration.

In [25]:
clusters_umap = clusterer(relations, umap=dict(n_components=5, n_neighbors=15, min_dist=0.0))

/home/rass/Desktop/projects/theoryminer/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


cluster: 1,197 spans -> 306 clusters | 180 noise (15%) | umap {'n_components': 5, 'n_neighbors': 15, 'min_dist': 0.0}


/home/rass/Desktop/projects/theoryminer/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


## 7. `standardize_groups` — groups to taxonomy concepts

```python
named = standardize_groups(groups, name_by="mean_sim", taxonomy="elsst", strategy="enriched", top=5, ...)
```

`mean_sim` averages the members' score vectors (conservative). `centroid` averages the vectors
first (scores higher). `medoid` scores the central member only. A low `mean_sim` score signals a
mixed group, or a construct the thesaurus lacks.

In [26]:
for rule in ("mean_sim", "centroid", "medoid"):
    named = standardize_groups(clusters, name_by=rule, top=3)
    multi = sorted(((k, v) for k, v in named.items() if v["size"] > 1), key=lambda kv: -kv[1]["size"])
    for k, v in multi[:4]:
        m = v["matches"][0]
        print(f"  {rule:<9} C{k} n={v['size']:<3} [{v['central']:<46}] -> {m['leaf']} ({m['score']:.2f})")
    print()
k, v = multi[0]
print("full record for one group:")
print({"central": v["central"], "size": v["size"], "members": v["members"][:4], "matches": v["matches"][:2]})

taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)


standardize: 710 groups named by mean_sim | best score: median 0.49, min 0.20, max 0.81
  mean_sim  C223 n=12  [higher intrinsic motivation                   ] -> MOTIVATION (0.60)
  mean_sim  C124 n=11  [procrastination                               ] -> SELF-DISCIPLINE (0.42)
  mean_sim  C240 n=11  [promoting learner autonomy                    ] -> LEARNING (0.46)
  mean_sim  C76 n=10  [distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful] -> DISTANCE EDUCATION (0.56)

taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)


standardize: 710 groups named by centroid | best score: median 0.51, min 0.20, max 0.83
  centroid  C223 n=12  [higher intrinsic motivation                   ] -> MOTIVATION (0.68)
  centroid  C124 n=11  [procrastination                               ] -> SELF-DISCIPLINE (0.49)
  centroid  C240 n=11  [promoting learner autonomy                    ] -> LEARNING (0.50)
  centroid  C76 n=10  [distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful] -> DISTANCE EDUCATION (0.69)

taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)


standardize: 710 groups named by medoid | best score: median 0.50, min 0.20, max 0.83
  medoid    C223 n=12  [higher intrinsic motivation                   ] -> INCENTIVES (0.65)
  medoid    C124 n=11  [procrastination                               ] -> SELF-DISCIPLINE (0.50)
  medoid    C240 n=11  [promoting learner autonomy                    ] -> LEARNING (0.46)
  medoid    C76 n=10  [distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful] -> DISTANCE EDUCATION (0.65)

full record for one group:
{'central': 'higher intrinsic motivation', 'size': 12, 'members': ['greater classroom motivation', "learner's intrinsic motivation was", 'intrinsic motivation', 'high intrinsic motivation'], 'matches': [{'id': 'https://elsst.cessda.eu/id/5/5d1671fa-2940-4d42-b993-599b3ed84e76', 'leaf': 'INCENTIVES', 'path': 'MOTIVATION > INCENTIVES', 'score': 0.6484}, {'id': 'https://elsst.cessda.eu/id/5/3cb85d5f-db5a-4222-8d5d-ef9d57d484ac', 'leaf': 'MO

### 7.3 `label_groups` — a label without a taxonomy

```python
labelled = label_groups(groups, by="keywords", top=3)     # keywords (the default) | medoid | both
```

No model, no taxonomy, no file. `medoid` is the `central` span the grouping already gives.
`keywords` ranks the content words of the member spans by class-based TF-IDF over the groups:
`(count in the group / words in the group) * log(1 + groups / groups that hold the word)`. The
first factor rewards a word the members repeat, the second punishes a word every group holds.
A group with one member takes the member as its label. The spans stay as they are.

In [27]:
for by in ("medoid", "keywords", "both"):
    labelled = label_groups(clusters, by=by, top=3)
    big = sorted(((k, v) for k, v in labelled.items() if v["size"] > 1), key=lambda kv: -kv[1]["size"])[:5]
    for k, v in big:
        print(f"  C{k} n={v['size']:<3} {v['name']}")
    print()
print("full record:", labelled[big[2][0]])
one = next(v for v in labelled.values() if v["size"] == 1)
print("a group of one:", one["name"])

labels: 710 groups labelled by medoid | 242 with 2 or more members
  C223 n=12  higher intrinsic motivation
  C124 n=11  procrastination
  C240 n=11  promoting learner autonomy
  C76 n=10  distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful
  C82 n=10  perceived competence

labels: 710 groups labelled by keywords | 242 with 2 or more members
  C223 n=12  motivation intrinsic learning
  C124 n=11  procrastination procrastinated procrastinate
  C240 n=11  autonomy learner promoting
  C76 n=10  distance learning teaching
  C82 n=10  competence perceived students'perceptions

labels: 710 groups labelled by both | 242 with 2 or more members
  C223 n=12  higher intrinsic motivation | motivation intrinsic learning
  C124 n=11  procrastination | procrastination procrastinated procrastinate
  C240 n=11  promoting learner autonomy | autonomy learner promoting
  C76 n=10  distance teaching requires ample preparation time and situation-spec

### 7.4 The two labels side by side

Where the thesaurus holds the construct its label is the better one: comparable across
studies, and it carries a definition. Where the thesaurus is far, the score says so and the
keyword label is the honest one. Report both.

In [28]:
elsst = {k: v["matches"][0] for k, v in standardize_groups(clusters, name_by="mean_sim", top=1).items()}
labelled = label_groups(clusters, by="keywords", top=3)
print()
for k, v in sorted(((k, v) for k, v in labelled.items() if v["size"] > 1), key=lambda kv: -kv[1]["size"])[:6]:
    m = elsst[k]
    print(f"  C{k} n={v['size']:<3} {v['name']:<40} ELSST: {m['leaf']} ({m['score']:.2f})")

taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)


standardize: 710 groups named by mean_sim | best score: median 0.49, min 0.20, max 0.81
labels: 710 groups labelled by keywords | 242 with 2 or more members

  C223 n=12  motivation intrinsic learning            ELSST: MOTIVATION (0.60)
  C124 n=11  procrastination procrastinated procrastinate ELSST: SELF-DISCIPLINE (0.42)
  C240 n=11  autonomy learner promoting               ELSST: LEARNING (0.46)
  C76 n=10  distance learning teaching               ELSST: DISTANCE EDUCATION (0.56)
  C82 n=10  competence perceived students'perceptions ELSST: PERSONAL EFFICACY (0.52)
  C139 n=9   poor self-rated health                   ELSST: HEALTH STATUS (0.47)


The functions `label_groups` and `standardize_groups` are taken apart line by line in
`tutorials/PACKAGE_LINE_BY_LINE.ipynb`, Part 3.

The same function on the pairwise groups gives the same shape.

In [29]:
named_pairwise = standardize_groups(groups, name_by="mean_sim", top=3)
k = next(k for k, v in named_pairwise.items() if 3 <= v["size"] <= 8)
named_pairwise[k]

taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)


standardize: 811 groups named by mean_sim | best score: median 0.49, min 0.16, max 0.79


{'central': 'differences',
 'size': 3,
 'members': ['differences between', 'difference', 'differences'],
 'matches': [{'id': 'https://elsst.cessda.eu/id/5/25f2d472-9e9f-417a-a187-6278e75d94f0',
   'leaf': 'AGE DIFFERENCES',
   'path': 'AGE GROUPS > GENERATIONS (AGE) > AGE DIFFERENCES',
   'score': 0.4381},
  {'id': 'https://elsst.cessda.eu/id/5/3c4cd600-4ff4-40ca-b69c-4e9362da9783',
   'leaf': 'CONFLICT',
   'path': 'CONFLICT',
   'score': 0.3616},
  {'id': 'https://elsst.cessda.eu/id/5/842d1098-3f00-4ada-ad3d-447b2108fce2',
   'leaf': 'REFUGEES',
   'path': 'MIGRANTS > REFUGEES',
   'score': 0.341}]}

## 8. Save what you want to keep

The functions return lists and dicts. `outputs/` is gitignored.

In [30]:
import csv
json.dump(matches, open("outputs/matches_enriched.json", "w"))
rows = [{"span": s, "group": g["group"], "central": g["central"]} for s, g in clusters.items()]
json.dump(rows, open("outputs/clusters.json", "w"))
with open("outputs/sentences_sample.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=data["sentences"][0].keys())
    w.writeheader(); w.writerows(data["sentences"])
print(sorted(os.listdir("outputs")))

['causal_map_full_run', 'causal_map_full_run.html', 'causal_map_round_2.html', 'causal_map_run', 'clusters.json', 'dois.csv', 'dois_full_run.csv', 'download_test', 'map_prototypes', 'matches_enriched.json', 'references.csv', 'references_full_run.csv', 'relations_sample.json', 'sentences_sample.csv', 'snowball_round_2_report.csv', 'snowball_round_3_dois.csv', 'snowball_test']


## 9. Traceability, end to end

```
doc_id ── page ── sent_id ── rel_id ── (cause span, effect span) ── span → node
```

The cell builds the edges of the pairwise groups by hand, the way `causal_map()` does inside
(section 10.8): rewrite each relation with the node of its cause and the node of its effect, merge
identical pairs, count. Then it walks the most supported edge back to its sentences.

In [31]:
node = {s: g["group"] for s, g in groups.items()}
central = {g["group"]: g["central"] for g in groups.values()}
edges = collections.defaultdict(list)
for r in relations:
    edges[(node[r["cause"]], node[r["effect"]])].append(r)
loops = sum(len(v) for (a, b), v in edges.items() if a == b)
print(f"{len(edges)} distinct edges from {len(relations):,} relations | {loops} relations became self-loops")
(a, b), support = max(((k, v) for k, v in edges.items() if k[0] != k[1]), key=lambda kv: len(kv[1]))
print()
print(f"edge  [{central[a]}] -> [{central[b]}]   weight {len(support)}   pages {sorted({r['page'] for r in support})}")
for r in support[:3]:
    print(f"  {r['rel_id']}  p{r['page']}  {r['cause']!r} -> {r['effect']!r}")
    print(f"      \"{r['sentence']}\"")

891 distinct edges from 945 relations | 9 relations became self-loops

edge  [social support] -> [addressing loneliness]   weight 6   pages [4, 16, 17]
  d005p0015s001r00  p4  'social support' -> 'individuals perceive feelings of loneliness as less stressful and more changeable'
      "Social support may help individuals perceive feelings of loneliness as less stressful and more changeable, thus buffering the possible effects on health."
  d005p0015s002r00  p4  'social support' -> 'mitigate maladaptive reactions to loneliness'
      "Additionally, social support may mitigate maladaptive reactions to loneliness (e.g., physical stress responses) and buffer possible health consequences."
  d005p0016s000r00  p4  'social support' -> 'moderating role in the relationship between loneliness and its health-related outcomes'
      "In summary, adapting the theoretical framework of the main effect and buffering model of social support [52], it can be suggested that social support may have a direc

## 10. A full run on the sample papers

This section runs the whole pipeline once, on all PDF files in `sample_files/`. Every cell runs for
real. No cell loads a saved file. The sections above show one function at a time. This section
shows how the results of all functions fit together.

The steps:

1. `harvest` reads the folder.
2. `extract_dois` finds the DOI and the title of each paper and saves `outputs/dois_full_run.csv`.
   `extract_references` finds the DOIs that the papers cite and saves `outputs/references_full_run.csv`.
3. `causenet` finds the cause-effect pairs. On CPU this takes about 2 minutes.
4. The harmonizer groups the spans and names the groups: with ELSST, and without a thesaurus.
5. The next cells show the results per paper with its DOI, the largest groups, and the strongest edges.
6. `causal_map` builds the map, `draw_map` shows it, and `save_map` writes its tables (section 10.8).

### 10.1 Harvest the folder

In [32]:
full = harvest("sample_files")

  reading .pdf file 3 of 7

  reading .pdf file 5 of 7

  reading .pdf file 7 of 7


harvest: sample_files
  files: 8 | read: 7 | skipped: 1
    skipped: SOURCES.md: the file name does not end with .pdf (not_pdf)
  text: 2,531 sentences kept | 1,865 rows dropped
       423  table_row
       328  no_lowercase
       247  reference_section
       229  too_short
       148  low_alpha
       107  running_head
       106  heading
        74  digit_heavy
        72  spaced_letters
        52  caption_or_keywords
        31  ref_fragment
        16  licence_text
        13  index_entry
         8  doi_url
         7  too_long
         2  empty_page
         1  page_number
         1  front_back_matter
  sentences per file:
       265  alrabai_2021_autonomy_supportive_teaching
       294  barrech_2018_job_insecurity_health
       296  friesinger_2025_discrimination_health
       384  gordesli_2024_social_media_mental_health
       318  pelikan_2021_needs_intrinsic_motivation
       611  schutz_2025_loneliness_social_support
       363  stephany_2017_income_inequality_trust


### 10.1b The DOI of each paper

The DOI links each `doc_id` to the published paper. Section 10.4 shows it next to the counts.

In [33]:
full_dois = extract_dois("sample_files", save="outputs/dois_full_run.csv")
doi_of = {}
for row in full_dois:
    doi_of[row["doc_id"]] = row["doi"]

  reading .pdf file 2 of 7

  reading .pdf file 4 of 7

  reading .pdf file 6 of 7

  reading .pdf file 7 of 7


dois: sample_files
  files: 8 | found: 7 | not found: 0 | skipped: 1
    not_pdf: SOURCES.md: the file name does not end with .pdf
  saved: outputs/dois_full_run.csv


### 10.1c The DOIs in the references

In [34]:
full_refs = extract_references("sample_files", save="outputs/references_full_run.csv")
shared = [row for row in full_refs if row["n_seeds"] > 1]
print()
print(f"{len(full_refs)} reference DOIs; cited by 2 or more seed papers: {[row['doi'] for row in shared]}")

  reading .pdf file 7 of 7
references: sample_files
  seeds read: 7 | unique reference DOIs: 215 | cited by 2 or more seeds: 1
       41  alrabai_2021_autonomy_supportive_teaching
        0  barrech_2018_job_insecurity_health
        3  friesinger_2025_discrimination_health
       50  gordesli_2024_social_media_mental_health
       42  pelikan_2021_needs_intrinsic_motivation
       74  schutz_2025_loneliness_social_support
        6  stephany_2017_income_inequality_trust
  no DOI in the references of: barrech_2018_job_insecurity_health (the journal printed none)
  skipped: SOURCES.md (the file name does not end with .pdf)
  saved: outputs/references_full_run.csv

215 reference DOIs; cited by 2 or more seed papers: ['10.1207/s15327965pli1104_01']


### 10.2 Find the cause-effect pairs

In [35]:
full_relations = causenet(full["sentences"])

  64/2,531 sentences (3s)

  128/2,531 sentences (6s)

  192/2,531 sentences (8s)

  256/2,531 sentences (12s)

  320/2,531 sentences (15s)

...

  2,304/2,531 sentences (109s)

  2,368/2,531 sentences (114s)

  2,432/2,531 sentences (116s)

  2,496/2,531 sentences (117s)

  2,531/2,531 sentences (118s)
causenet: 490 causal sentences of 2,531 | 945 relations
  spans repaired after decoding: 124 | both spans found verbatim in their sentence: 97%
  skipped (avoid_ambiguous=True): 44 relations with a pointer span, for example: these differences -> models being more comprehensive; powerful enough to disclose the net effect size of the treatment manipulated in this study -> this; it -> impacts individuals'health and well-being


### 10.3 Group and name the spans

Four calls. Each one takes the output of the call before it, or the relations.

In [36]:
full_matches  = standardize_constructs(full_relations, taxonomy="elsst", leaves_only=False, top=5)
full_groups   = pairwise_grouping(full_relations, threshold=0.69)
full_clusters = clusterer(full_relations)
full_named    = standardize_groups(full_clusters, name_by="mean_sim", top=5)
full_labelled = label_groups(full_clusters)                  # the TF-IDF keywords (the default)

taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)


standardize: 1,197 spans -> top 5 of 3,739 entries | best score: median 0.51, min 0.16, max 0.83


pairwise: 1,197 spans -> 811 groups at cos >= 0.69, average linkage | 180 groups with 2+ members, largest 27


/home/rass/Desktop/projects/theoryminer/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


cluster: 1,197 spans -> 242 clusters | 468 noise (39%) | raw embeddings
taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)


standardize: 710 groups named by mean_sim | best score: median 0.49, min 0.20, max 0.81
labels: 710 groups labelled by keywords | 242 with 2 or more members


### 10.4 Results per paper

The DOI, the count of kept sentences and the count of relations for each paper. A paper with more causal claims in
its text gives more relations per sentence.

In [37]:
sentences_per_doc = collections.Counter(r["doc_id"] for r in full["sentences"])
relations_per_doc = collections.Counter(r["doc_id"] for r in full_relations)

print(f"{'paper':<44} {'DOI':<30} {'sentences':>9} {'relations':>9}")
for doc_id in sorted(sentences_per_doc):
    print(f"{doc_id:<44} {doi_of.get(doc_id, ''):<30} {sentences_per_doc[doc_id]:>9} {relations_per_doc[doc_id]:>9}")
print(f"{'all':<44} {'':<30} {len(full['sentences']):>9} {len(full_relations):>9}")

paper                                        DOI                            sentences relations
alrabai_2021_autonomy_supportive_teaching    10.3389/fpsyg.2021.728657            265       155
barrech_2018_job_insecurity_health           10.1186/s12889-018-5621-4            294        99
friesinger_2025_discrimination_health        10.1186/s12889-025-23888-6           296       105
gordesli_2024_social_media_mental_health     10.1371/journal.pone.0316365         384       209
pelikan_2021_needs_intrinsic_motivation      10.1371/journal.pone.0257346         318       121
schutz_2025_loneliness_social_support        10.1186/s12889-025-23247-5           611       185
stephany_2017_income_inequality_trust        10.1007/s11205-016-1460-9            363        71
all                                                                              2531       945


Three relations from each paper, with the `rel_id` and the page:

In [38]:
for doc_id in sorted(sentences_per_doc):
    print(doc_id)
    doc_relations = [r for r in full_relations if r["doc_id"] == doc_id]
    for r in doc_relations[:3]:
        print(f"   [{r['rel_id']} p{r['page']}] {r['cause']!r} -> {r['effect']!r}")

alrabai_2021_autonomy_supportive_teaching
   [d000p0003s003r00 p1] 'experimental intervention' -> 'efl ehsan namaziandost'
   [d000p0006s003r00 p1] 'desire' -> 'autonomy'
   [d000p0006s003r01 p1] 'desire' -> 'take charge'
barrech_2018_job_insecurity_health
   [d001p0004s001r00 p1] 'changing economic environment' -> 'organizational change'
   [d001p0004s001r01 p1] 'changing economic environment' -> 'reorganizations'
   [d001p0004s001r02 p1] 'changing economic environment' -> 'working environments'
friesinger_2025_discrimination_health
   [d002p0002s000r00 p1] 'factors' -> 'experience discrimination'
   [d002p0002s000r01 p1] 'age, gender, illness' -> 'experience discrimination'
   [d002p0002s000r02 p1] 'disability' -> 'experience discrimination'
gordesli_2024_social_media_mental_health
   [d003p0006s001r00 p1] 'popularity of social media' -> 'time adolescents spent information files'
   [d003p0006s002r00 p1] 'on social media' -> 'social media use influences the mental funding'
   [d003p0

### 10.5 The largest groups, with two names

`label_groups` names a cluster from its own words. `standardize_groups` names it with the closest
ELSST concept and a score. A low score means that ELSST has no good concept for the group.

In [39]:
multi = [(k, v) for k, v in full_labelled.items() if v["size"] > 1]
multi.sort(key=lambda kv: -kv[1]["size"])

print(f"{len(multi)} clusters with 2 or more spans; the 10 largest:")
for k, v in multi[:10]:
    best = full_named[k]["matches"][0]
    print(f"  C{k:<4} n={v['size']:<3} {v['name'][:58]:<58} ELSST: {best['leaf']} ({best['score']:.2f})")

242 clusters with 2 or more spans; the 10 largest:
  C223  n=12  motivation intrinsic learning                              ELSST: MOTIVATION (0.60)
  C124  n=11  procrastination procrastinated procrastinate               ELSST: SELF-DISCIPLINE (0.42)
  C240  n=11  autonomy learner promoting                                 ELSST: LEARNING (0.46)
  C76   n=10  distance learning teaching                                 ELSST: DISTANCE EDUCATION (0.56)
  C82   n=10  competence perceived students'perceptions                  ELSST: PERSONAL EFFICACY (0.52)
  C139  n=9   poor self-rated health                                     ELSST: HEALTH STATUS (0.47)
  C197  n=7   trust generalized level                                    ELSST: TRUST (0.65)
  C235  n=7   efl learner autonomy                                       ELSST: EDUCATIONAL EXPECTATIONS (0.40)
  C15   n=6   internet access smartphone                                 ELSST: INTERNET ACCESS (0.43)
  C20   n=6   appreciates carefu

### 10.6 The strongest edges, by hand

This cell shows the idea of the map in a few lines. Each span becomes the label of its cluster. A
noise span keeps its own text. Each relation then becomes an edge from the cause node to the effect
node. The weight of an edge is the number of relations behind it. Section 10.8 does the same with
`causal_map()`, which also counts papers, flags two-way pairs and self-loops, and filters.

In [40]:
def node_of(span):
    """Return the node label of a span: its cluster label, or the span for a noise span."""
    group = full_clusters[span]["group"]
    if group == -1:
        return full_labelled[span]["name"]
    return full_labelled[group]["name"]


edges = collections.defaultdict(list)
for r in full_relations:
    cause_node = node_of(r["cause"])
    effect_node = node_of(r["effect"])
    if cause_node != effect_node:
        edges[(cause_node, effect_node)].append(r)

strongest = sorted(edges.items(), key=lambda kv: -len(kv[1]))
print(f"{len(edges):,} distinct edges from {len(full_relations):,} relations; the 8 strongest:")
for (cause_node, effect_node), support in strongest[:8]:
    papers = sorted({r["doc_id"].split("_")[0] for r in support})
    print(f"  weight {len(support):<3} [{cause_node[:40]}] -> [{effect_node[:40]}]   papers: {papers}")

873 distinct edges from 945 relations; the 8 strongest:
  weight 4   [racism] -> [inequities health produces]   papers: ['friesinger']
  weight 4   [motivation intrinsic learning] -> [procrastination procrastinated procrasti]   papers: ['pelikan']
  weight 3   [insecurity job uncertainty] -> [employees unhealthy health]   papers: ['barrech']
  weight 3   [media social using] -> [mental health problems]   papers: ['gordesli']
  weight 3   [individualism scored horizontal] -> [depression levels lower]   papers: ['gordesli']
  weight 3   [competence perceived students'perception] -> [motivation intrinsic learning]   papers: ['pelikan']
  weight 3   [loneliness] -> [psychosomatic complaints multiple]   papers: ['schutz']
  weight 3   [loneliness] -> [mental health problems]   papers: ['schutz']


One edge, walked back to its sentences:

In [41]:
(cause_node, effect_node), support = strongest[0]
print(f"edge [{cause_node}] -> [{effect_node}], weight {len(support)}")
for r in support[:3]:
    print(f"  {r['rel_id']}  {r['doc_id']}  p{r['page']}")
    print(f"     {r['cause']!r} -> {r['effect']!r}")
    print(f"     \"{r['sentence']}\"")

edge [racism] -> [inequities health produces], weight 4
  d002p0006s014r00  friesinger_2025_discrimination_health  p2
     'racism' -> 'societal inequities'
     "The consequences of racism include societal inequities and adverse health outcomes, such as poorer physical and mental health [13, 19]."
  d002p0007s003r00  friesinger_2025_discrimination_health  p2
     'racism' -> 'produces ethnic health inequalities'
     "With this epidemiological lens, racism produces ethnic health inequalities that manifest as health inequities through specific pathways [3, 9]."
  d002p0007s003r01  friesinger_2025_discrimination_health  p2
     'racism' -> 'health inequities'
     "With this epidemiological lens, racism produces ethnic health inequalities that manifest as health inequities through specific pathways [3, 9]."


### 10.7 Checks after the run

Four numbers to read after every run:

- the spans in both roles: a chain in the graph passes only through these;
- the largest pairwise group: the chaining check (section 5.1);
- the share of spans with an ELSST score at or above 0.55;
- the noise share of `clusterer`.

In [42]:
causes = {r["cause"] for r in full_relations}
effects = {r["effect"] for r in full_relations}
pairwise_sizes = collections.Counter(g["group"] for g in full_groups.values())
at_tau = sum(m[0]["score"] >= 0.55 for m in full_matches.values()) / len(full_matches)
noise = sum(g["group"] == -1 for g in full_clusters.values()) / len(full_clusters)

print("spans in both roles       :", len(causes & effects))
print("largest pairwise group    :", max(pairwise_sizes.values()))
print(f"ELSST score >= 0.55       : {at_tau:.0%}")
print(f"noise share of clusterer  : {noise:.0%}")

spans in both roles       : 41
largest pairwise group    : 27
ELSST score >= 0.55       : 35%
noise share of clusterer  : 39%


### 10.8 The causal map

```python
cmap = causal_map(relations, names, min_papers=1, min_sentences=2, min_share=0.67)
draw_map(cmap, papers=None, save=None, title="Causal map", height=720)
save_map(cmap, folder)
```

`names` is the output of `label_groups`, `standardize_groups` or `standardize_constructs`, made from
the same relations. A node is a group or a thesaurus concept. An edge goes from the node of the cause
to the node of the effect, with three counts: papers, sentences and relations. One sentence can give two
relations on one edge, for example "social media → body image" and "social media → beauty standards" when
both effects are in one group; they count as one sentence. Every edge keeps its `rel_id`s, so it leads
back to its sentences.

- A **self-loop** (the cause and the effect in one node) is flagged. It usually means that the grouping
  merged two constructs that a paper keeps apart.
- A **two-way pair** (A → B and B → A) keeps both edges. When one side has at least `min_share` of the
  papers, it is `dominant` and the other `minority` (dashed on the page). Else both are `contested`
  (two colours on the page).
- The **filters** hide the edges below `min_papers` or `min_sentences`. A hidden edge goes to
  `cmap["hidden"]` with its `drop_reason`. Nothing is deleted. `min_sentences=2` hides the claims that
  only one sentence makes; use 1 to show every edge.
- The **role** of a node comes from the shown edges: "cause only", "cause and effect", "effect only".

`draw_map` shows the page in the notebook, or writes `causal_map.html` from a script. On the page:
hover over a node or an edge, click an edge for all its sentences (with the DOI, else the title, else
the file name), drag, zoom, and save a picture as PNG, JPG or SVG. The search is a filter: "age" shows
only the nodes whose label or member spans have a word that starts with "age", their neighbours, and the
edges between them. More filters: one paper, the roles, two-way pairs only, `min papers`,
`min sentences` and self-loops. Reset brings back the filters of `causal_map`. The
page gets every edge, so its sliders can show the hidden edges again. `save_map` writes `nodes.csv`,
`edges.csv`, `hidden.csv` and `map.graphml` (for Gephi or Cytoscape desktop).

In [43]:
from theoryminer import causal_map, draw_map, save_map

# The clusterer groups, named by label_groups (section 10.3). The DOIs and titles come from 10.1b.
full_map = causal_map(full_relations, full_labelled)
draw_map(full_map, papers=full_dois, save="outputs/causal_map_full_run.html", title="Causal map: sample papers")
save_map(full_map, "outputs/causal_map_full_run")

causal map: 50 nodes | 35 edges shown, 849 hidden | self-loops: 11 edges, 12 of 945 relations (1.3%) | two-way pairs: 6 (6 contested)
saved page: outputs/causal_map_full_run.html


saved map: outputs/causal_map_full_run (nodes.csv, edges.csv, hidden.csv, map.graphml)


['outputs/causal_map_full_run/nodes.csv',
 'outputs/causal_map_full_run/edges.csv',
 'outputs/causal_map_full_run/hidden.csv',
 'outputs/causal_map_full_run/map.graphml']

The strongest shown edges, and one edge walked back to its sentences:

In [44]:
def edge_weight(edge):
    """Return the sort key of an edge: papers first, then sentences."""
    return (edge["n_papers"], edge["n_sentences"])


nodes = full_map["nodes"]
for edge in sorted(full_map["edges"], key=edge_weight, reverse=True)[:6]:
    print(f"  {edge['n_papers']} papers, {edge['n_sentences']} sentences  "
          f"[{nodes[edge['source']]['label'][:35]}] -> [{nodes[edge['target']]['label'][:35]}]  {edge['direction']}")

edge = max(full_map["edges"], key=edge_weight)
print()
print(f"edge {edge['edge_id']}: {nodes[edge['source']]['label']} -> {nodes[edge['target']]['label']}")
for rel_id in edge["rel_ids"][:3]:
    relation = full_map["relations"][rel_id]
    print(f"  {rel_id}  {relation['doc_id']}  p{relation['page']}")
    print(f"     {relation['cause']!r} -> {relation['effect']!r}")
    print(f"     \"{relation['sentence']}\"")

  2 papers, 2 sentences  [autonomy self-reported constructs] -> [motivation intrinsic learning]  one-way
  1 papers, 4 sentences  [motivation intrinsic learning] -> [procrastination procrastinated proc]  one-way
  1 papers, 3 sentences  [insecurity job uncertainty] -> [employees unhealthy health]  one-way
  1 papers, 3 sentences  [racism] -> [inequities health produces]  one-way
  1 papers, 3 sentences  [media social using] -> [mental health problems]  one-way
  1 papers, 3 sentences  [individualism scored horizontal] -> [depression levels lower]  one-way

edge e00120: autonomy self-reported constructs -> motivation intrinsic learning
  d000p0033s003r00  alrabai_2021_autonomy_supportive_teaching  p11
     'autonomy' -> 'intrinsic motivation'
     "This finding also confirms that autonomy-supportive environment is strongly connected with intrinsic motivation in that in autonomy supportive contexts, intrinsic motivation is maintained and enhanced, whereas in controlled contexts, intrinsi

The same relations with the three kinds of names. Only the names change; the relations stay.

In [45]:
for route, names in [("pairwise + label_groups", label_groups(full_groups)),
                     ("clusterer + label_groups", full_labelled),
                     ("standardize_constructs (ELSST)", full_matches)]:
    print(route)
    causal_map(full_relations, names)
    print()

labels: 811 groups labelled by keywords | 180 with 2 or more members
pairwise + label_groups
causal map: 47 nodes | 38 edges shown, 853 hidden | self-loops: 6 edges, 9 of 945 relations (1.0%) | two-way pairs: 7 (7 contested)

clusterer + label_groups
causal map: 50 nodes | 35 edges shown, 849 hidden | self-loops: 11 edges, 12 of 945 relations (1.3%) | two-way pairs: 6 (6 contested)

standardize_constructs (ELSST)
causal map: 47 nodes | 58 edges shown, 761 hidden | self-loops: 17 edges, 40 of 945 relations (4.2%) | two-way pairs: 21 (21 contested)



## 11. A snowball round: from one seed paper to a new map

A snowball search starts from seed papers and follows their references. This section runs one round:

1. `extract_references` finds the DOIs in the references of one seed paper.
2. `download_papers` downloads the open-access PDFs of those DOIs.
3. The pipeline of section 10 reads the new papers and draws their map.
4. `extract_references` on the new papers gives the DOIs for the next round.

```python
report = download_papers(dois, output_dir, licences=None, email=None, core_api_key=None,
                         serpapi_key=None, save=None)
```

`dois` is the output of `extract_references`, a list of DOIs, or a CSV file with a `doi` column.
The function asks OpenAlex (free, no key) for the licence and the PDF links of each paper. With
`licences=["cc-by"]` it downloads only the CC BY papers: their text may be shown and shared, as the text
of `sample_files/`. Then it tries the OpenAlex links, and then `tmsr-doi-downloader`. `pip install theoryminer` installs
it on Python 3.11 or newer; on Python 3.10 the function uses OpenAlex only. The settings `email` (for
Unpaywall), `core_api_key` (free, from core.ac.uk) and `serpapi_key` (paid) switch on more sources of
that tool. This run uses none of them.

Each DOI gets one row with a `status`: `downloaded`, `already_there`, `skipped_licence`, `not_found`
or `error`. A failed DOI does not stop the run. Many publishers block robots, so not every paper can be
downloaded. The function reads `robots.txt`, waits 1 second between requests, and keeps a file only when
it is a real PDF. A second run does not download a file again. This section needs an internet connection.

### 11.1 The references of one seed paper

In [46]:
from theoryminer import download_papers

SEED = "sample_files/schutz_2025_loneliness_social_support.pdf"
ROUND_2 = "outputs/snowball_round_2"          # the PDFs of the next round; git ignores outputs/

seed_refs = extract_references(SEED)

  reading .pdf file 1 of 1
references: sample_files/schutz_2025_loneliness_social_support.pdf
  seeds read: 1 | unique reference DOIs: 74 | cited by 2 or more seeds: 0
       74  schutz_2025_loneliness_social_support


### 11.2 Download the CC BY papers

In [47]:
report = download_papers(seed_refs, ROUND_2, licences=["cc-by"], save="outputs/snowball_round_2_report.csv")
print()
for row in report[:8]:
    print(f"  {row['status']:<15} {row['licence']:<10} {row['source']:<20} {row['doi']}")

  downloading 1 of 74

  downloading 2 of 74

  downloading 3 of 74

  downloading 4 of 74

  downloading 5 of 74

...

  downloading 71 of 74

  downloading 72 of 74

  downloading 73 of 74

  downloading 74 of 74


download: 74 DOIs -> outputs/snowball_round_2 in 4.4 min
  downloaded: 14 | skipped_licence: 45 | not_found: 15
  downloaded from: {'openalex': 11, 'tmsr-doi-downloader': 3}
  saved: outputs/snowball_round_2_report.csv

  skipped_licence other-oa                        10.1002/per.2220
  skipped_licence none                            10.1002/pits.20576
  skipped_licence none                            10.1002/wps.20839
  not_found       cc-by                           10.1007/978-3-319-47143-3_12
  skipped_licence none                            10.1007/978-3-662-50345-4_46
  not_found       cc-by                           10.1007/s00103-021-03291-3
  not_found       cc-by                           10.1007/s00103-023-03728-x
  not_found       cc-by                           10.1007/s10389-024-02356-2


### 11.3 The map of the new papers

The same pipeline as in section 10, on the downloaded papers. The names come from `pairwise_grouping`
(average linkage) and `label_groups`, the second route next to the clusterer map of section 10.8.
The file name of each PDF is its DOI, so `extract_dois` finds the DOI and the title of each paper.

In [48]:
round_data = harvest(ROUND_2)
round_papers = extract_dois(ROUND_2)
round_relations = causenet(round_data["sentences"])
round_names = label_groups(pairwise_grouping(round_relations))
round_map = causal_map(round_relations, round_names)
draw_map(round_map, papers=round_papers, save="outputs/causal_map_round_2.html",
         title="Causal map: the CC BY references of one seed paper")

  reading .pdf file 2 of 14

  reading .pdf file 5 of 14

  reading .pdf file 7 of 14

  reading .pdf file 10 of 14

  reading .pdf file 13 of 14

...

  4,288/4,450 sentences (232s)

  4,352/4,450 sentences (235s)

  4,416/4,450 sentences (238s)

  4,450/4,450 sentences (240s)
causenet: 802 causal sentences of 4,450 | 1,555 relations
  spans repaired after decoding: 246 | both spans found verbatim in their sentence: 94%
  skipped: 2 relations with an empty span after decoding
  skipped (avoid_ambiguous=True): 64 relations with a pointer span, for example: variables -> those are affected; school social experiences are taken into -> those are affected; such action -> make school a significant setting for interventions that reduce students'loneliness


pairwise: 1,868 spans -> 1,165 groups at cos >= 0.69, average linkage | 277 groups with 2+ members, largest 81
labels: 1,165 groups labelled by keywords | 277 with 2 or more members
causal map: 72 nodes | 66 edges shown, 1,365 hidden | self-loops: 10 edges, 13 of 1555 relations (0.8%) | two-way pairs: 15 (15 contested)
saved page: outputs/causal_map_round_2.html


### 11.4 The next round

The new papers are the seeds of the next round. The DOIs that several of them cite come first.

In [49]:
next_refs = extract_references(ROUND_2, save="outputs/snowball_round_3_dois.csv")
print()
for row in next_refs[:5]:
    print(f"  cited by {row['n_seeds']} papers: {row['doi']}")

  reading .pdf file 14 of 14
references: outputs/snowball_round_2
  seeds read: 14 | unique reference DOIs: 472 | cited by 2 or more seeds: 26
        2  10_1007_s10578-022-01449-x
       55  10_1007_s10964-021-01441-z
        2  10_1017_s0954579421001632
       49  10_1111_bjc_12331
       44  10_1111_bjep_12616
      112  10_1177_10731911221077227
       74  10_1371_journal_pone_0251750
       35  10_1371_journal_pone_0279701
        7  10_25646_11868
        1  10_25646_11876
        1  10_25646_11878
       62  10_3389_fpubh_2023_1275917
       45  10_3389_fpubh_2024_1392999
       13  10_3390_ijerph181910420
  saved: outputs/snowball_round_3_dois.csv

  cited by 3 papers: 10.1007/s12160-010-9210-8
  cited by 3 papers: 10.1016/j.jaac.2020.05.009
  cited by 3 papers: 10.1037/dev0000117
  cited by 3 papers: 10.1177/1403494819865429
  cited by 2 papers: 10.1002/pits.20120


## 12. Settings at a glance

| function | setting | default | source of the default |
|---|---|---|---|
| `harvest` | `min_words` / `max_words` | 4 / 80 | old cleaning notebook |
| `extract_dois` | `first_pages` / `save` | 2 / `None` | the paper's own DOI is on page 1 (checked by hand: 7 of 7) |
| `extract_references` | `save` / `cache` | `None` / `True` | 214 of 215 DOIs of the sample papers exist at doi.org |
| `download_papers` | `licences` | `None` | `["cc-by"]` keeps only CC BY papers (section 11) |
| `download_papers` | `email` / `core_api_key` / `serpapi_key` | `None` | switch on Unpaywall, CORE and Google Scholar in `tmsr-doi-downloader`; `None` keeps the environment variable |
| `causenet` | `threshold` | 0.8 | old pipeline sweep 0.5–0.8 |
| `causenet` | `mode` | `neural` | model card default |
| `causenet` | `decision` | `span_only` | since 0.2.0 (`cls+span` before): a cause span and an effect span are enough |
| `causenet` | `avoid_ambiguous` | `True` | since 0.2.0: skip a relation with a pointer span ("this", "it") |
| `causenet` | `batch_size` / `device` | 64 / `auto` | `auto` = an NVIDIA GPU, else an Apple GPU, else the CPU |
| all harmonizer | `embeddings` | `allmpnet` | Paper 3; `bge_base` scores 0.02 F1 higher (`STRATEGIES.md` section 5) |
| `standardize_*` | `taxonomy` | `elsst` | project choice; the only built-in thesaurus |
| `standardize_*` | `strategy` | `enriched` | Paper 3 web tool; confirmed by the held-out test |
| `standardize_*` | `top` | 5 | project choice |
| `standardize_*` | `leaves_only` | `False` | since 0.2.0 (`True` before): broad concepts too, such as TRUST and MOTIVATION |
| `label_groups` | `by` / `top` | `keywords` / 3 | since 0.2.0 (`both` before): the TF-IDF keywords of the group |
| `standardize_*` | `threshold` | `None` | since 0.2.0; `None` gives every span its best concept |
| `standardize_groups` | `name_by` | `mean_sim` | old pipeline's cluster-naming comparison |
| `pairwise_grouping` | `threshold` | 0.69 | Paper 3 `BEST_PAIRWISE_THRESHOLDS` |
| `pairwise_grouping` | `linkage` | `average` | since 0.2.0 (`single` before): single linkage chains (section 5.1) |
| `clusterer` | `min_cluster_size` / `min_samples` | 2 / 2 | Paper 3 `BEST_CLUSTERING_PARAMS` |
| `clusterer` | `umap` | `None` | Paper 3; the best run had no reduction |
| `causal_map` | `min_papers` / `min_sentences` | 1 / 2 | project choice; 1 paper works for one long book |
| `causal_map` | `min_share` | 0.67 | project choice: a two-way pair needs 2 papers to 1 for one direction |
| `draw_map` | `save` / `height` | `None` / 720 | a notebook shows the page; a script writes `causal_map.html` |

τ, the score that a span needs for a thesaurus concept, is the `threshold` of the `standardize_*`
functions. It is off by default. Choose it per model, per strategy, per thesaurus.
The evidence behind the defaults is in `tutorials/STRATEGIES.md` section 5.